# GSE211713 | Mouse-lung radiation-associated state shifts

**Public application gallery — ScGeo 0.2.0.** Reuse the checksum-matched corrected v2 representations and the original compact and annotated H5ADs for 131,157 retained cells. The primary cross-sectional contrasts are control, early 17 Gy (months 1/2) and late 17 Gy (months 4/5). The 20-library dataset includes other doses/time groups; the primary gallery focuses on the 13 mice in these three groups. No individual mouse is followed longitudinally.

We retain the original approved major and fibroblast states, the 30/20 cells-per-mouse eligibility cutoffs and the minimum of three eligible mice per group. The stored original mouse-centered estimates are not refitted. The source reports and CSV checksums are verified; group membership and cell counts are cross-checked. See the [application guide](README.md).

In [ ]:
from pathlib import Path
import sys
import os
import numpy as np
import pandas as pd
import scgeo as sg
from IPython.display import display

cwd = Path.cwd().resolve()
root = next((p for p in (cwd, *cwd.parents)
             if (p / "scripts/scgeo_v020_applications.py").is_file()), None)
if root is None:
    raise RuntimeError("Open from the companion repository checkout.")
sys.path.insert(0, str(root / "scripts"))
from scgeo_v020_applications import Application, PRIMARY, OTHER, install_graph, plot_evidence
app = Application("C")
adata, annotation = app.load()
print(adata.shape, list(adata.obsm))

## 1. Define primary groups, retain the complete denominator

Use all retained cells in the selected primary mice for abundance and graph context, including excluded/unapproved states. Source geometry remains qualified state-by-mouse. Display downsampling is not analytical downsampling.

In [ ]:
dose = pd.to_numeric(adata.obs["dose_gy"]).astype(float)
month = pd.to_numeric(adata.obs["month_post_irradiation"]).astype(float)
adata.obs["application_group"] = np.select(
    [dose.eq(0), dose.eq(17) & month.isin([1, 2]), dose.eq(17) & month.isin([4, 5])],
    ["control", "17Gy_early", "17Gy_late"], default="other_dose_time")
adata = adata[adata.obs["application_group"].ne("other_dose_time")].copy()
annotation = annotation.loc[adata.obs_names]
groups = ["control", "17Gy_early", "17Gy_late"]
assert adata.obs["mouse_id"].notna().all()
assert adata.obs.groupby("mouse_id", observed=True)["application_group"].nunique().le(1).all()
roster = adata.obs.groupby("application_group", observed=True)["mouse_id"].nunique().reindex(groups)
assert roster.tolist() == [5, 4, 4]
display(roster)
roster.to_csv(app.out / "primary_mouse_roster.csv")
major_states = app.config["states"]["major"]
fibro_states = app.config["states"]["fibroblast"]
basis = "umap_display_only"
def display_labels(key, approved):
    values = adata.obs[key].astype("string")
    return pd.Categorical(values.where(values.isin(approved), OTHER).fillna(OTHER),
                          categories=[*approved, OTHER])
adata.obs["application_state"] = display_labels("state_major", major_states)
adata.obs["application_fibroblast"] = display_labels("fibroblast_state", fibro_states)
counts = pd.crosstab([adata.obs["application_group"], adata.obs["mouse_id"]],
                     adata.obs["application_state"], dropna=False)
# crosstab(dropna=False) can contain nonobserved group-by-mouse combinations; remove them.
counts = counts.loc[counts.sum(axis=1).gt(0)]
counts.to_csv(app.out / "mouse_state_counts_all_retained.csv")
fractions = counts.div(counts.sum(axis=1), axis=0)
fractions.to_csv(app.out / "mouse_state_fractions_all_retained.csv")
display(fractions)
app.save(sg.pl.embedding_facets(adata, basis=basis, facet="application_group",
    facet_order=groups, color="application_state", shared_limits=True, col_wrap=3,
    max_points=12000, point_size=2, random_state=1729, title="GSE211713 | Primary radiation groups",
    return_result=True, show=False), "primary_condition_facets",
    "Common display-only UMAP; all retained cells in primary mice")

## 2. Fibroblast and marker-score views

The fibroblast-focused facet is a deliberate visual subset, not the abundance denominator. The separate score view uses the stored annotation scores; it does not prove a molecular driver.

In [ ]:
fibro = adata[adata.obs["application_fibroblast"].astype(str).isin(fibro_states)].copy()
app.save(sg.pl.embedding_facets(fibro, basis=basis, facet="application_group",
    facet_order=groups, color="application_fibroblast", shared_limits=True, col_wrap=3,
    max_points=10000, point_size=3, random_state=1729, title="GSE211713 | Approved fibroblast states",
    return_result=True, show=False), "fibroblast_facets",
    "Fibroblast display subset only; never use this subset as total retained-cell denominator")
del fibro
for score in ["major_score_fibroblast_stromal", "refined_score_myofibroblast"]:
    adata.obs[score] = annotation[score].to_numpy()
    app.save(sg.pl.embedding_facets(adata, basis=basis, facet="application_group",
        facet_order=groups, color=score, shared_limits=True, col_wrap=3, cmap="viridis",
        max_points=10000, point_size=2, random_state=1729, title=f"GSE211713 | {score} (stored)",
        return_result=True, show=False), score, "Stored descriptive annotation score")
app.save(sg.pl.embedding_density(adata, "application_group", basis=basis, groups=groups,
    normalize="per_group", gridsize=80, contour=True, smooth_k=3, log1p=True,
    return_result=True, show=False), "relative_density", "Relative embedding density, not sample abundance")

## 3. Use the existing v2 PCA50 cell graph

The original representation object includes a sparse PCA50 neighbor graph. Restrict it to the full retained-cell primary roster and derive symmetric state adjacency by aggregating edge weights; this is **new descriptive graph aggregation**, not recomputed PAGA or an original statistical result. The same graph and display basis are used for the primary contrasts. The normalized edge-mass formula and graph origins are exported.

In [ ]:
assert "connectivities_pca50" in adata.obsp
graph = adata.obsp["connectivities_pca50"]
names = install_graph(app, adata, graph, "major",
    "Frozen v2 connectivities_pca50, induced subgraph on all 13 primary mice")
display(adata.uns["application_adjacency"]["connectivities"])

## 4. Major-state response across all three primary contrasts

The original per-contrast source report checksums and qualified mouse IDs/counts must agree. Preserve unapproved display states as context without inventing evidence for them. Biological sample intervals are read in the same normalized units as the effects. Individual mouse-center plots show the first PCA50 coordinate (not a full displacement or a new test); each point is one eligible mouse, with no invented pairing.

In [ ]:
for contrast, group0, group1 in PRIMARY:
    frame, centers = app.c_evidence(adata, contrast, group0, group1, "major")
    report = plot_evidence(app, adata, frame, group0, group1, "application_group",
                           basis, f"major_{contrast}", names, sample="mouse_id")
    centers.to_csv(app.out / f"major_{contrast}_pca50_pc1_mouse_centers.csv", index=False)
    app.save(sg.pl.replicate_effect(centers, coordinate=0, condition_order=[group0, group1],
        summary=None, state_order=major_states, title=f"{group0} vs {group1} | Eligible mouse centers, PC1",
        return_result=True, show=False), f"major_{contrast}_mouse_centers",
        "Frozen qualified mouse centers; PC1 is one coordinate only; cross-sectional, unpaired")
    subset = adata[adata.obs["application_group"].isin([group0, group1])].copy()
    app.save(sg.pl.geometry_dynamics_atlas(subset, basis=basis, color="application_state",
        node_key="application_state", condition_key="application_group", group0=group0,
        group1=group1, sample_key="mouse_id", panels=("states", "evidence"), report=report,
        effect_metric="normalized_delta_norm", max_points=8000, random_state=1729,
        title=f"{group0} vs {group1} | Geometry-only evidence atlas", show=False),
        f"major_{contrast}_atlas", "Frozen sample-aware evidence and display-only states; no dynamics")
    del subset

## 5. Refined fibroblast-state response

Keep all non-target cells as an explicit context node to retain the complete denominator for sample abundance. The quantitative report contains only the three prespecified fibroblast states. The remaining cells are not an inferred common biological state and have no stored shift estimate under this hierarchy.

In [ ]:
adata.obs["application_state"] = adata.obs["application_fibroblast"].copy()
names = install_graph(app, adata, graph, "fibroblast",
    "Same frozen PCA50 graph; other retained cells collapsed into a display context label")
for contrast, group0, group1 in PRIMARY:
    frame, centers = app.c_evidence(adata, contrast, group0, group1, "fibroblast")
    report = plot_evidence(app, adata, frame, group0, group1, "application_group",
                           basis, f"fibroblast_{contrast}", names, sample="mouse_id")
    centers.to_csv(app.out / f"fibroblast_{contrast}_pca50_pc1_mouse_centers.csv", index=False)
    app.save(sg.pl.replicate_effect(centers, coordinate=0, condition_order=[group0, group1],
        summary=None, state_order=fibro_states, title=f"{group0} vs {group1} | Fibroblast mouse centers, PC1",
        return_result=True, show=False), f"fibroblast_{contrast}_mouse_centers",
        "Frozen eligible mouse centers; PC1 only; no pairing")

## 6. Provenance and limitations

No original graph or representation file was altered. No RNA velocity, temporal cell tracking, OOD score or exact signed gene attribution was manufactured. The saved expression and representations have no PCA loadings; further molecular mapping verification is a separate task. The frozen secondary contrasts remain in the original reproduction workflow. These application figures do not replace the original analysis or constitute new biological validation.

In [ ]:
manifest = app.finish()
display({k: manifest[k] for k in ["status", "dataset", "plot_manifests", "figures", "source_tables"]})